# 02 — Method B: INT4 **PTQ** (TorchAO, 사후 양자화)

Method A의 머지 BF16 모델을 입력으로, **추가 학습 없이** weight-only INT4로 사후 양자화한다. 01과 **동일한 5-파트 템플릿·동일 eval**을 사용해 3-way 표의 B 행을 채운다.

> ## ⚙️ 실행 모드 배너 — 이 노트북은 **Azure A100 80GB에서 실제 실행**됨 (프로덕션/스펙 경로)
>
> **컴퓨트:** Azure `Standard_NC24ads_A100_v4` (**NVIDIA A100 80GB PCIe** ×1) · region **japaneast** ·
> `compute.mode: gpu`. MCAPS 거버넌스가 온디맨드 GPU SKU 배포를 차단하므로 **Spot 우선순위**로 프로비저닝.
>
> **양자화(스펙 B):** 입력 = `artifacts/A_bf16/` (A의 머지 BF16), TorchAO `Int4WeightOnlyConfig`
> (weight-only, **group_size=128**, **tile-packed / tinygemm** tensor-core INT4 커널, bf16 compute).
> `lm_head`·임베딩은 bf16 유지(표준 weight-only 서빙 포맷). 학습 단계 **없음**.
>
> **재현:** 버전 고정값은 `results/env_B.json`, 수치는 `results/B_int4_ptq_metrics.json`,
> 3-way 표는 `results/three_way_table.json`(B 행)에 기록된다.

### 이 방법(B) — 무엇/왜/어떻게
- **무엇:** post-training quantization(PTQ). 이미 학습된 가중치를 그대로 INT4로 변환.
- **왜:** 재학습 비용 0 — 가장 빠르고 단순한 압축. 품질 손실이 양자화 오차에 직접 노출되는 baseline.
- **어떻게(C와의 차이):** C(QAT)는 학습 중 양자화를 시뮬레이션해 오차를 보정하지만, B는 보정 없음.
  B·C는 **동일한 INT4 서빙 포맷**(tile-packed g128)으로 저장하므로, 차이는 순수하게 *train-aware* 여부.

### 0) 부트스트랩 & 버전 고정 (재현성)
`quantization/` 공용 모듈을 import하고 정확한 버전을 `results/env_B.json`에 기록.

In [1]:
import os, sys
here = os.getcwd()
for cand in [here, os.path.dirname(here), os.path.join(here, "pdf_qa_extraction"),
             os.path.dirname(os.path.dirname(here))]:
    if os.path.isdir(os.path.join(cand, "quantization")):
        if cand not in sys.path:
            sys.path.insert(0, cand)
        os.chdir(cand)
        break
print("cwd:", os.getcwd())

cwd: /home/azureuser/work/pdf_qa_extraction


In [2]:
import json, platform, torch, transformers, torchao
env = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "torchao": torchao.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"),
}
os.makedirs("quantization/results", exist_ok=True)
with open("quantization/results/env_B.json", "w", encoding="utf-8") as fh:
    json.dump(env, fh, ensure_ascii=False, indent=2)
env

{'python': '3.10.12',
 'torch': '2.11.0+cu130',
 'transformers': '5.5.0',
 'torchao': '0.17.0',
 'cuda_available': True,
 'device': 'NVIDIA A100 80GB PCIe'}

### 1) 설정 로드 + A 머지 확인
`config.yaml`의 `ptq.group_size`를 사용. 입력인 A 머지 모델이 선행 저장돼 있어야 한다.

In [3]:
from quantization.data_korquad import load_config, load_korquad
import quantization.eval_qa as E
cfg = load_config()
A_DIR = cfg['paths']['method_a_dir']
B_DIR = cfg['paths']['method_b_dir']
gs = int(cfg['ptq']['group_size'])
assert os.path.isdir(A_DIR), f'A 머지 모델 없음: {A_DIR} (먼저 01 실행)'
print('입력 A 머지 :', A_DIR)
print('출력 B(int4):', B_DIR)
print('group_size  :', gs, '| base:', cfg['base_model']['selected'])

입력 A 머지 : quantization/artifacts/A_bf16
출력 B(int4): quantization/artifacts/B_int4_ptq
group_size  : 128 | base: Qwen/Qwen3-1.7B


### 2) 데이터 — KorQuAD (A/B/C 동일 held-out)
eval 슬라이스와 perplexity 컨텍스트는 A와 **완전히 동일**(seed·크기 고정).

In [4]:
data = load_korquad(cfg)
print('train:', len(data['train']), '| eval:', len(data['eval']))

train: 60407 | eval: 500


### 3) INT4 PTQ 양자화 (TorchAO tile-packed)
A 머지 로드 → `TorchAoConfig(Int4WeightOnlyConfig tile-packed)`로 양자화 → 저장. 학습 없음.

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TorchAoConfig
tok = AutoTokenizer.from_pretrained(A_DIR)
if tok.pad_token is None: tok.pad_token = tok.eos_token
qmodel = AutoModelForCausalLM.from_pretrained(
    A_DIR, dtype=torch.bfloat16, device_map='cuda',
    quantization_config=TorchAoConfig(quant_type=E.make_int4_weightonly_config(gs)))
os.makedirs(B_DIR, exist_ok=True)
qmodel.save_pretrained(B_DIR); tok.save_pretrained(B_DIR)
size_gb = E.dir_size_gb(B_DIR)
print(f'INT4 PTQ 저장 완료: {B_DIR}  size={size_gb:.3f} GB')
# free the working model so VRAM below reflects the reloaded serving artifact only
del qmodel; torch.cuda.empty_cache()

INT4 PTQ 저장 완료: quantization/artifacts/B_int4_ptq  size=1.288 GB


### 4) 동작 데모 (필수) — held-out 질문 **1개**
디스크에 저장한 INT4 서빙 아티팩트를 **새로 로드**해 실제로 답을 생성(“동작한다”를 눈으로).

In [6]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(B_DIR, device_map='cuda')
demo = data['eval'][0]
gen = E.generate_answers(model, tok, [demo.prompt], max_new_tokens=32, batch_size=1)
print('[질문]', demo.question)
print('[정답]', demo.answers)
print('[모델 답]', gen['answers'][0])

[질문] 2004년 이명박이 서울시장 재직시절 전면적으로 개선한 것은?
[정답] ['대중교통체계']
[모델 답] 대중교통체계


### 5) 수치 — EM/F1 · perplexity · 크기 · VRAM · tok/s
재로드한 INT4 모델로 A와 **동일한 eval**을 수행 → `results/B_int4_ptq_metrics.json` 저장 + 3-way 표 B 행 append.

In [7]:
res = E.evaluate_model(model, tok, data['eval'], method='B_int4_ptq',
                       base_model=cfg['base_model']['selected'],
                       model_dir=B_DIR,
                       max_new_tokens=cfg['eval']['max_new_tokens'],
                       batch_size=cfg['eval']['batch_size'],
                       ppl_samples=cfg['eval']['ppl_samples'],
                       precision='int4',
                       notes=f'TorchAO int4 PTQ tile-packed g{gs} (no retrain)')
E.write_metrics(res, cfg['paths']['results_dir'])
E.append_to_table(res, cfg['paths']['results_dir'])
from dataclasses import asdict
row = asdict(res)
print('B (INT4 PTQ) — 3-way 표 B 행')
for k in ['method','base_model','exact_match','f1','perplexity','size_gb','peak_vram_gb','tok_per_s','precision']:
    print(f'  {k:14}: {row[k]}')

B (INT4 PTQ) — 3-way 표 B 행
  method        : B_int4_ptq
  base_model    : Qwen/Qwen3-1.7B
  exact_match   : 65.2
  f1            : 80.692
  perplexity    : 15.9039
  size_gb       : 1.2878
  peak_vram_gb  : 4.5849
  tok_per_s     : 37.39
  precision     : int4


### 6) 서빙 & 다음 단계
저장된 INT4 아티팩트(`artifacts/B_int4_ptq/`)는 TorchAO tile-packed 포맷으로, transformers에서 재로드해 즉시 서빙 가능(위 데모가 이를 증명). 동일 포맷을 **C(QAT)** 가 재사용하므로 `03` 실행 후 3-way 표(A/B/C)가 완성되고, vLLM INT4 서빙 벤치로 잇는다.